# Setup

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!pip install -q bitsandbytes>=0.46.1 peft trl accelerate transformers datasets openai scikit-learn jsonlines pyyaml huggingface-hub

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path
import os, sys
import json

### Directory Setup from Google Drive & Repo

In [5]:
RUN_ID = "deduplicated_v2"

DRIVE_ROOT = Path("/content/drive/MyDrive/latent_safety_probing")

RUN_ROOT = DRIVE_ROOT / "runs" / RUN_ID

INTERIM_DIR = RUN_ROOT / "data" / "interim"
DATA_DIR = (
    RUN_ROOT
    / "data"
    / "processed"
    / "beavertails_risk_v2"
)
CKPT_DIR = RUN_ROOT / "checkpoints"
RESULTS_DIR = RUN_ROOT / "results"

for directory in [
    INTERIM_DIR,
    DATA_DIR,
    CKPT_DIR,
    RESULTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Run root:", RUN_ROOT)
print("Interim data:", INTERIM_DIR)
print("Processed data:", DATA_DIR)
print("Checkpoints:", CKPT_DIR)

Run root: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2
Interim data: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/data/interim
Processed data: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/data/processed/beavertails_risk_v2
Checkpoints: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/checkpoints


In [6]:
import subprocess

REPO_URL = (
    "https://github.com/khadijahslawal/latent-watch.git"
)
CODE_DIR = Path("/content/latent-watch")

if (CODE_DIR / ".git").exists():
    # This handles rerunning the cell in the same Colab session.
    subprocess.run(
        ["git", "-C", str(CODE_DIR), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(CODE_DIR), "checkout", "main"],
        check=True,
    )
    subprocess.run(
        [
            "git",
            "-C",
            str(CODE_DIR),
            "pull",
            "--ff-only",
            "origin",
            "main",
        ],
        check=True,
    )
elif CODE_DIR.exists():
    raise RuntimeError(
        f"{CODE_DIR} exists but is not a Git repository. "
        "Inspect it before continuing."
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(CODE_DIR),
        ],
        check=True,
    )

commit = subprocess.run(
    ["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", CODE_DIR)
print("Commit:", commit)

Repository: /content/latent-watch
Commit: b6bb2f9


## Imports & API Keys Env

In [7]:
import os
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(CODE_DIR / "requirements.txt"),
    ],
    check=True,
)

os.chdir(CODE_DIR)

source_dir = str(CODE_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

print("Working directory:", Path.cwd())

Working directory: /content/latent-watch


In [8]:
from google.colab import userdata

os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Training

In [9]:
# Shared hyperparameters — edit here
TRAINING_CFG = dict(
    model_name_or_path          = 'meta-llama/Llama-3.2-1B-Instruct',
    load_in_4bit                = True,
    fp16                        = True,   # T4: True | A100: False
    lora_r                      = 16,
    lora_alpha                  = 32,
    lora_dropout                = 0.05,
    target_modules              = ['q_proj', 'v_proj'],
    num_train_epochs            = 3,
    per_device_train_batch_size = 4,      # reduce to 4 if OOM - prev 8
    gradient_accumulation_steps = 32,      #up from  4
    learning_rate               = 2e-4,
    warmup_ratio                = 0.05,
    max_seq_length              = 512,
    seed                        = 42,
)

In [10]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

E1_ADAPTER = CKPT_DIR / "answer_only" / "best_adapter"
E2_ADAPTER = CKPT_DIR / "cot" / "best_adapter"
E3_ADAPTER = CKPT_DIR / "coconut" / "best_adapter"

In [11]:
required_adapters = {
    "E1": E1_ADAPTER,
    "E2": E2_ADAPTER,
    "E3": E3_ADAPTER,
}

for name, path in required_adapters.items():
    print(f"\n{name}: {path}")
    assert path.exists(), f"Missing directory: {path}"
    assert (path / "adapter_config.json").exists()
    assert (
        (path / "adapter_model.safetensors").exists()
        or (path / "adapter_model.bin").exists()
    )
    print("Adapter files present")


E1: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/checkpoints/answer_only/best_adapter
Adapter files present

E2: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/checkpoints/cot/best_adapter
Adapter files present

E3: /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/checkpoints/coconut/best_adapter
Adapter files present


**Review Formatters First**

In [16]:
# Load one row from each split directory
# ao_row  = json.loads(open(f"{DATA_DIR}/answer_only/train.jsonl").readline())
# cot_row = json.loads(open(f"{DATA_DIR}/cot/train.jsonl").readline())

In [15]:
# from src.training.formatters import format_answer_only, format_cot

# inp, tgt = format_answer_only(ao_row)
# print("=== E1 answer_only ===")
# print("INPUT:\n", inp[:400])
# print("TARGET:", tgt)

In [17]:
# inp, tgt = format_cot(cot_row)
# print("\n=== E2 cot ===")
# print("INPUT:\n", inp[:400])
# print("TARGET:\n", tgt[:300])

## Experiment 1 - Answer only baseline

### Answer Only Test Evaluation

In [12]:
from src.evaluation.evaluate_classification import evaluate as evaluate_text

e1_results = evaluate_text(
    experiment="answer_only",
    adapter_dir=E1_ADAPTER,
    dataset_dir=DATA_DIR / "answer_only",
    output_file=RESULTS_DIR / "answer_only_test.csv",
    model_name=MODEL_NAME,
    load_in_4bit=True,
    fp16=True,  # appropriate for a Colab T4
    max_new_tokens=32,
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

...

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Evaluating 358 test examples [answer_only]...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Results → /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/results/answer_only_test.csv

Test Evaluation - answer_only
Examples evaluated : 358
Valid-output rate  : 1.0000
Accuracy           : 0.7291
Weighted Precision : 0.7460
Weighted Recall    : 0.7291
Weighted F1        : 0.7243  ← PRIMARY METRIC

HIGH_RISK Recall   : 0.8603  ← KEY (costly to miss unsafe prompts)
LOW_RISK  Recall   : 0.5978

Full Classification Report:
              precision    recall  f1-score   support

   HIGH_RISK       0.68      0.86      0.76       179
    LOW_RISK       0.81      0.60      0.69       179

    accuracy                           0.73       358
   macro avg       0.75      0.73      0.72       358
weighted avg       0.75      0.73      0.72       358

Confusion Matrix (rows=true, cols=pred):
                 HIGH_RISK     LOW_RISK
     HIGH_RISK           154            25
      LOW_RISK            72           107


## CoT Test Evaluation

In [13]:
# E2 — same
e2_results = evaluate_text(
    experiment="cot",
    adapter_dir=E2_ADAPTER,
    dataset_dir=DATA_DIR / "cot",
    output_file=RESULTS_DIR / "cot_test.csv",
    model_name=MODEL_NAME,
    load_in_4bit=True,
    fp16=True,
    max_new_tokens=256,
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Evaluating 358 test examples [cot]...
Results → /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/results/cot_test.csv

Test Evaluation - cot
Examples evaluated : 358
Valid-output rate  : 1.0000
Accuracy           : 0.7039
Weighted Precision : 0.7158
Weighted Recall    : 0.7039
Weighted F1        : 0.6998  ← PRIMARY METRIC

HIGH_RISK Recall   : 0.8212  ← KEY (costly to miss unsafe prompts)
LOW_RISK  Recall   : 0.5866

Full Classification Report:
              precision    recall  f1-score   support

   HIGH_RISK       0.67      0.82      0.73       179
    LOW_RISK       0.77      0.59      0.66       179

    accuracy                           0.70       358
   macro avg       0.72      0.70      0.70       358
weighted avg       0.72      0.70      0.70       358

Confusion Matrix (rows=true, cols=pred):
                 HIGH_RISK     LOW_RISK
     HIGH_RISK           147            32
      LOW_RISK            74           105


## Coconut Test Evaluation

Confirm that E3 must contains the extended tokenizer

In [14]:
from transformers import AutoTokenizer

e3_tokenizer = AutoTokenizer.from_pretrained(E3_ADAPTER)

bot_id = e3_tokenizer.convert_tokens_to_ids("<bot>")
eot_id = e3_tokenizer.convert_tokens_to_ids("<eot>")

print("<bot>:", bot_id)
print("<eot>:", eot_id)

assert bot_id != e3_tokenizer.unk_token_id
assert eot_id != e3_tokenizer.unk_token_id

<bot>: 128256
<eot>: 128257


In [15]:
from src.evaluation.evaluate_coconut_classification import (
    evaluate as evaluate_latent
)

e3_results = evaluate_latent(
    experiment="latent",
    adapter_dir=E3_ADAPTER,
    dataset_dir=DATA_DIR / "cot",
    output_file=RESULTS_DIR / "coconut_test.csv",
    model_name=MODEL_NAME,
    load_in_4bit=True,
    fp16=True,
    max_new_tokens=64,
    num_latent_steps=3,
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Evaluating 358 test examples [latent]...
Results → /content/drive/MyDrive/latent_safety_probing/runs/deduplicated_v2/results/coconut_test.csv

Test Evaluation - latent
Examples evaluated : 358
Valid-output rate  : 1.0000
Accuracy           : 0.7263
Weighted Precision : 0.7304
Weighted Recall    : 0.7263
Weighted F1        : 0.7250  ← PRIMARY METRIC

HIGH_RISK Recall   : 0.7933  ← KEY (costly to miss unsafe prompts)
LOW_RISK  Recall   : 0.6592

Full Classification Report:
              precision    recall  f1-score   support

   HIGH_RISK       0.70      0.79      0.74       179
    LOW_RISK       0.76      0.66      0.71       179

    accuracy                           0.73       358
   macro avg       0.73      0.73      0.73       358
weighted avg       0.73      0.73      0.73       358

Confusion Matrix (rows=true, cols=pred):
                 HIGH_RISK     LOW_RISK
     HIGH_RISK           142            37
      LOW_RISK            61           118
